# Reproducing Zhang et al. (2023)

Zhang, R., Chen, Y., Lei, J., Zhou, X., Yao, P., Stive, M.J.F. (2023). *Experimental
investigation of wave attenuation by mangrove forests with submerged canopies.*
Coastal Engineering 186, 104403. doi:[10.1016/j.coastaleng.2023.104403](https://doi.org/10.1016/j.coastaleng.2023.104403)

The experiment: a 1:10 model patch of 36 artificial mangroves (2 m × 0.5 m) in
configurations M1 (stem + roots), M2 (+ sparse canopy) and M3 (+ dense canopy), at
depths of 0.3, 0.4 and 0.5 m, under regular waves. This notebook rebuilds the paper's
geometry, hydraulic ranges, drag closures and damping factors with SADIE.

```{important}
Using Zhang et al.'s own drag-coefficient fits to predict Zhang et al.'s own damping
factors is a **consistency check, not independent validation**: it tests whether
SADIE reproduces the paper's definitions and analysis, not whether the physics
predicts new data. Independent validation is a later milestone.
```

Page numbers refer to the journal pages of the paper. Equation numbers "theory
Eq. x" refer to the [theory](../theory.md) page.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np

import sadiepy as sd
from sadiepy.presets.zhang2023 import FIG4_BETA, REPORTED_RANGES, TEST_MATRIX

# Many cases lie outside a closure's fitted range (recorded in result.extrapolated),
# and three wave conditions exceed linear-theory limits (listed in section 2).
# Silence the repeated warnings to keep the notebook readable.
warnings.simplefilter("ignore", sd.ValidityWarning)  # includes ExtrapolationWarning

water = sd.Water(temperature=20.0)  # the paper does not state the water temperature
VEG = {c: sd.presets.zhang2023(c) for c in ("M1", "M2", "M3")}
COLOR = {"M1": "#2a78d6", "M2": "#eb6834", "M3": "#1baf7a"}  # validated categorical slots
MARKER = {"M1": "o", "M2": "s", "M3": "^"}
N_TREES = 36.0
plt.rcParams.update(
    {
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "lines.linewidth": 2,
    }
)

## 1. Geometry: Fig. 3

Zhang et al., Fig. 3 (p. 6), plots per tree, against water depth: the projected area
$A_p$; the shares of the canopy and of the cylindrical part (stem + roots) in it; the
submerged volume SV; and the submerged volume fraction SVF. The caption gives
$A_p$ and SV as formulas; the preset builds the same quantities from layers
(theory §5.1).

**Erratum E7, as resolved.** The caption's canopy term $0.5\,W_{cs}h_{cs}$ is the same for
M2 and M3, but Fig. 3a shows M3 with about twice the canopy area of M2. The preset
doubles the canopy frontal area for M3, which reproduces Fig. 3a and 3b.

In [ ]:
h = np.linspace(0.01, 0.6, 240)
fig, ax = plt.subplots(1, 4, figsize=(13, 4), sharey=True)
for c, veg in VEG.items():
    a_p = np.array([veg.frontal_area(x) for x in h]) / N_TREES
    a_canopy = (
        np.array([36 * veg.layers[2].frontal_area(x) for x in h]) / N_TREES
        if c != "M1"
        else np.zeros_like(h)
    )
    sv = np.array([veg.submerged_volume(x) for x in h]) / N_TREES
    svf = np.array([veg.svf(x) for x in h])
    kw = {"color": COLOR[c], "label": c}
    ax[0].plot(a_p, h, **kw)
    ax[1].plot(a_canopy / a_p, h, **kw)
    ax[1].plot(1 - a_canopy / a_p, h, ls="--", color=COLOR[c])
    ax[2].plot(sv * 1e4, h, **kw)
    ax[3].plot(svf, h, **kw)
ax[0].set(xlabel="$A_p$ per tree (m²)", ylabel="water depth $h$ (m)", title="(a) projected area")
ax[1].set(xlabel="area fraction", title="(b) canopy (solid), cylinders (dashed)")
ax[2].set(xlabel="SV per tree ($10^{-4}$ m³)", title="(c) submerged volume")
ax[3].set(xlabel="SVF", title="(d) submerged volume fraction")
ax[0].legend()
fig.tight_layout()
plt.show()

for c, veg in VEG.items():
    print(
        f"{c}: A_p(h=0.5) = {veg.frontal_area(0.5) / N_TREES:.4f} m², "
        f"SV(h=0.5) = {veg.submerged_volume(0.5) / N_TREES:.3e} m³, "
        f"SVF at h = 0.3/0.4/0.5: " + ", ".join(f"{veg.svf(x):.4f}" for x in (0.3, 0.4, 0.5))
    )

The SVF over the tested cases spans 0.0062–0.0168, the range printed in Fig. 10
(p. 8). The values of $A_p$ at $h = 0.5$ m match the plateaus of Fig. 3a.

## 2. Hydraulic ranges

Zhang et al. report ranges of the hydraulic parameters for the 28 wave conditions
of Table 2 (p. 5). Re and KC use the depth-averaged velocity $\bar u = \omega a/(kh)$
(Zhang Eq. 8; theory Eq. 2.9) and the stem diameter $d$ (DBH = 0.01 m), or the
vegetation hydraulic radius $r_v$ (theory Eq. 5.3), which depends on the
configuration and depth.

In [ ]:
def ranges(velocity):
    vals = {k: [] for k in ("kh", "H/L", "H/h", "Ur", "Re_d", "KC_d", "Re_rv")}
    for t in TEST_MATRIX:
        w = sd.RegularWave(t.H, t.T, t.h)
        u = velocity(w)
        vals["kh"].append(w.kh)
        vals["H/L"].append(w.steepness)
        vals["H/h"].append(w.relative_height)
        vals["Ur"].append(w.ursell)
        vals["Re_d"].append(u * 0.01 / water.nu)
        vals["KC_d"].append(u * w.T / 0.01)
        for c in t.configs:
            if c != "M0":
                vals["Re_rv"].append(u * VEG[c].hydraulic_radius(t.h) / water.nu)
    return {k: (min(v), max(v)) for k, v in vals.items()}


u_bar = ranges(lambda w: w.u_bar)
u_swl = ranges(lambda w: float(w.u_m(w.depth)))  # orbital velocity at the still-water level

print(f"{'quantity':10s} {'SADIE, u_bar':>22s} {'SADIE, u_m at SWL':>22s}   reported (source)")
for key, rep in REPORTED_RANGES.items():
    base = key.split(" ")[0]
    a, b = u_bar[base], u_swl[base]
    cols = f"{a[0]:10.4g} – {a[1]:<10.4g} {b[0]:10.4g} – {b[1]:<10.4g}"
    print(f"{key:10s} {cols}  {rep[0]:.4g} – {rep[1]:.4g}")

### Linear-theory validity

SADIE checks every wave against the limits of linear theory (theory §2.10). Three of
the 28 conditions exceed the second-order nonlinearity criterion (Dean & Dalrymple 1991, Eq. 11.41):

In [ ]:
flagged = []
for t in TEST_MATRIX:
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", sd.ValidityWarning)
        w = sd.RegularWave(t.H, t.T, t.h)
    if caught:
        flagged.append(t)
        print(
            f"H = {t.H} m, T = {t.T} s, h = {t.h} m: Ur = {w.ursell:.1f}, ka = {w.k * w.a:.3f} "
            f"> {w.trough_bump_limit:.3f}"
        )
# Results for these three conditions are computed but carry this caveat.

- kh, H/L, H/h and Ur reproduce the reported ranges.
- **KC_d (erratum E5).** With $\bar u$, KC_d spans 6.05–64.89, the range in Table 3
  (p. 12), which is the range the closures were fitted on. The Conclusions' range
  (12.97–77.39) and their Re_d range (1.28–4.27 × 10³) are reproduced instead with the
  orbital velocity at the still-water level, $u_m(h) = a\omega/\tanh kh$. The paper
  therefore used two velocity scales; the closures use $\bar u$.
- Re_rv matches Table 3 (34 900–407 000) to within about 0.5 %. The small offset is
  consistent with a slightly different viscosity: the water temperature is not stated.

## 3. The published drag closures: Table 3, Figs. 13 and 15

Each closure is drawn as a solid line over its fitted range and as a dotted line
outside it. Re_d closures come from experiments at different scales (1:1 to 1:12).
Following Zhang §4.4 (after Kelty et al. 2022), they are compared at prototype scale,
$\mathrm{Re}_\text{prototype} = \mathrm{Re}_\text{source}\,\lambda^{3/2}$ (theory Eq. 5.6).

In [ ]:
from sadiepy.scaling.froude import rescale_reynolds

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]


def draw(ax, names, x_of_source, label_x):
    # x_of_source maps a source-scale value to the plotted value
    for i, name in enumerate(names):
        c = sd.drag.get(name)
        lo, hi = c.valid_range
        lo = max(lo, 1e-3 * hi)
        xs_in = np.geomspace(lo, hi, 200)
        xs_out = np.geomspace(lo / 3, hi * 3, 400)
        col = PALETTE[i % len(PALETTE)]
        ax.plot([x_of_source(c, x) for x in xs_out], [c(x) for x in xs_out], ":", color=col, lw=1)
        ax.plot(
            [x_of_source(c, x) for x in xs_in],
            [c(x) for x in xs_in],
            color=col,
            label=f"{name} (1:{c.scale:g})",
        )
    ax.set(xscale="log", ylim=(0, 40), xlabel=label_x, ylabel="$C_D$")
    ax.legend(fontsize=8)


same = lambda c, x: x  # noqa: E731
to_proto = lambda c, x: rescale_reynolds(x, c.scale, 1.0)  # noqa: E731
fig, ax = plt.subplots(2, 3, figsize=(14, 8), sharey=True)
draw(ax[0, 0], ["he2019_kc", "chang2022_kc", "bryant2022_kc", "zhang2023_kc"], same, "$KC_d$")
draw(
    ax[0, 1], ["maza2017_re", "chang2022_re", "kelty2022_re"], to_proto, "$Re_d$ at prototype scale"
)
draw(ax[0, 2], ["wang2022b_kc"], same, "$KC_{rv}$")
draw(ax[1, 0], ["zhang2023_rerv"], same, "$Re_{rv}$ (model scale)")
draw(ax[1, 1], ["zhang2023_ur"], same, "$Ur$")
ax[1, 2].set_visible(False)
fig.tight_layout()
plt.show()

## 4. Consistency check: damping factors of Fig. 4

Zhang et al. obtained each $C_D$ by inverting their Eq. (4), which uses a
**depth-averaged** frontal area $\bar D = \frac1h\int A\,dz$ (theory Eq. 5.5). The
closures were fitted to those $C_D$ values with Re, KC and Ur computed for the
incident wave. A consistent prediction therefore uses the same geometry and evaluates
the closure once at $H_0$:

```python
sd.propagate(wave, veg.depth_averaged(h), B, drag="zhang2023_kc", cd_evaluation="incident")
```

The measured $\beta$ values are those printed in the panels of Fig. 4 (p. 7).

In [ ]:
CLOSURES = ["zhang2023_kc", "zhang2023_rerv", "zhang2023_ur"]
B = 2.0
rows = []
for (H, T, h), measured in FIG4_BETA.items():
    wave = sd.RegularWave(H, T, h)
    for c, beta_meas in measured.items():
        geom = VEG[c].depth_averaged(h)
        row = {
            "case": (H, T, h),
            "config": c,
            "measured": beta_meas,
            "beta_unit_cd": sd.propagate(wave, geom, B, water=water, drag=1.0).beta,
        }
        for name in CLOSURES:
            r = sd.propagate(wave, geom, B, water=water, drag=name, cd_evaluation="incident")
            row[name] = r.beta
            row["cd_" + name] = r.drag_coefficients["depth-averaged"][0]
        row["cd_back"] = beta_meas / row["beta_unit_cd"]  # C_D that reproduces the measured beta
        row["KC_d"] = wave.u_bar * T / 0.01
        rows.append(row)

print(f"{'H, T, h':16s} cfg  measured   kc    rerv    ur  | C_D back  C_D(kc)")
for r in rows:
    H, T, h = r["case"]
    print(
        f"{H:.2f}, {T:.1f}, {h:.1f}   {r['config']}  {r['measured']:7.3f} "
        + " ".join(f"{r[n]:6.3f}" for n in CLOSURES)
        + f" | {r['cd_back']:7.2f}  {r['cd_zhang2023_kc']:7.2f}"
    )

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 4.2), sharex=True, sharey=True)
for a, name in zip(ax, CLOSURES, strict=True):
    for c in ("M1", "M2", "M3"):
        sel = [r for r in rows if r["config"] == c]
        a.scatter(
            [r["measured"] for r in sel],
            [r[name] for r in sel],
            s=60,
            color=COLOR[c],
            marker=MARKER[c],
            edgecolor="white",
            linewidth=1,
            label=c,
            zorder=3,
        )
    a.plot([0.005, 2], [0.005, 2], color="0.5", lw=1)
    a.set(xscale="log", yscale="log", title=name, xlabel=r"measured $\beta$ (1/m)")
    ratios = np.array([r[name] / r["measured"] for r in rows])
    by_cfg = {
        c: np.exp(np.mean(np.log([r[name] / r["measured"] for r in rows if r["config"] == c])))
        for c in ("M1", "M2", "M3")
    }
    print(
        f"{name:15s} geometric-mean predicted/measured: "
        + ", ".join(f"{c} {v:.2f}" for c, v in by_cfg.items())
        + f"; all {np.exp(np.mean(np.log(ratios))):.2f}"
    )
ax[0].set_ylabel(r"predicted $\beta$ (1/m)")
ax[0].legend()
fig.tight_layout()
plt.show()

**Reading the result.** The three Zhang closures were each fitted to M1, M2 and M3
pooled together. For a given wave, $KC_d$, $Re_d$ and Ur are identical for all three
configurations, so a pooled closure on these variables gives **the same $C_D$ to M1
and M3**. The predicted difference between configurations then comes from geometry
alone: the frontal area of M3 is only 1.25–2.4 times that of M1. The measured $C_D$
differ much more (back-calculated: about 1–2 for M1, 4–9 for M3, in line with the
paper's M1 mean of 2.63 and M2/M3 range of 1.78–29.7, p. 10). So the pooled closures
over-predict M1 several-fold and are closest for M3, which dominates the fit. The
$Re_{rv}$ closure separates the configurations partly, because $r_v$ shrinks as the
canopy raises the SVF.

The back-calculated $C_D$ of the canopy models fall close to the $KC_d$ curve:

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4))
kc = np.linspace(6, 65, 200)
ax.plot(kc, [sd.drag.get("zhang2023_kc")(x) for x in kc], color="0.3", label="zhang2023_kc")
for c in ("M1", "M2", "M3"):
    sel = [r for r in rows if r["config"] == c]
    ax.scatter(
        [r["KC_d"] for r in sel],
        [r["cd_back"] for r in sel],
        s=60,
        color=COLOR[c],
        marker=MARKER[c],
        edgecolor="white",
        linewidth=1,
        label=f"{c} (back-calculated)",
        zorder=3,
    )
ax.set(xlabel="$KC_d$", ylabel="$C_D$", ylim=(0, 15))
ax.legend(fontsize=8)
plt.show()

## 5. Qualitative targets

The paper's headline ratios (§4.1, p. 7) are computed from the Fig. 4 values. They
are compared here with the ratios predicted by each closure in the consistent
geometry.

In [ ]:
def beta(cfg, case, source):
    return next(r[source] for r in rows if r["config"] == cfg and r["case"] == case)


TARGETS = [
    ("M3/M1, H 0.08, T 1.4, h 0.3", "M3", "M1", (0.08, 1.4, 0.3), 7.8),
    ("M3/M1, H 0.08, T 1.6, h 0.3", "M3", "M1", (0.08, 1.6, 0.3), 6.2),
    ("M3/M1, H 0.08, T 1.4, h 0.4", "M3", "M1", (0.08, 1.4, 0.4), 10.7),
] + [
    (f"M3/M2, H {k[0]}, T {k[1]}, h {k[2]}", "M3", "M2", k, None)
    for k in FIG4_BETA
    if "M2" in FIG4_BETA[k]
]
print(f"{'target':32s} {'paper':>6s} {'Fig.4':>6s} " + " ".join(f"{n[9:]:>6s}" for n in CLOSURES))
for label, a, b, case, paper in TARGETS:
    meas = beta(a, case, "measured") / beta(b, case, "measured")
    pred = [beta(a, case, n) / beta(b, case, n) for n in CLOSURES]
    print(
        f"{label:32s} {paper if paper else '≈2':>6} {meas:6.2f} "
        + " ".join(f"{p:6.2f}" for p in pred)
    )

print("\nLargest beta by depth (H 0.08 m, T 1.4 s), measured and with zhang2023_kc:")
for c in ("M2", "M3"):
    cases = [k for k in FIG4_BETA if k[:2] == (0.08, 1.4) and c in FIG4_BETA[k]]
    meas = max(cases, key=lambda k: beta(c, k, "measured"))[2]
    pred = max(cases, key=lambda k: beta(c, k, "zhang2023_kc"))[2]
    print(f"  {c}: measured h = {meas}, predicted h = {pred}")

## 6. Contributions of canopy, stem and roots: Fig. 8 and erratum E4

Zhang et al. split $\beta$ into canopy, stem and root parts with their Eqs. (19)–(20)
(p. 9). Each part is Eq. (4) over the component's submerged height with its
**layer-averaged** width, and one $C_D$ for the whole plant. This is
`Vegetation.layer_averaged(h)` (theory Eq. 5.4), whose layer fractions give the
ratios $\beta_{rc} = \beta_c/\beta_s$ and $\beta_{rr} = \beta_r/\beta_s$ directly. The
paper states that above Ur ≈ 10 they approach 3.7 and 1.4.

In [ ]:
DEPTH_COLOR = {0.3: "#2a78d6", 0.4: "#eb6834", 0.5: "#1baf7a"}
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for c, filled in (("M2", True), ("M3", False)):
    for t in TEST_MATRIX:
        w = sd.RegularWave(t.H, t.T, t.h)
        f = sd.propagate(w, VEG[c].layer_averaged(t.h), B, drag=1.0).layer_fractions
        col = DEPTH_COLOR[t.h]
        style = {
            "s": 40,
            "edgecolor": col,
            "linewidth": 1.5,
            "zorder": 3,
            "facecolor": col if filled else "none",
        }
        ax[0].scatter(w.ursell, f["canopy"] / f["stem"], **style)
        if filled:  # beta_r / beta_s does not depend on the canopy
            ax[1].scatter(w.ursell, f["root"] / f["stem"], **style)
for h_, col in DEPTH_COLOR.items():
    ax[0].scatter([], [], s=40, color=col, label=f"h = {h_} m, M2 canopy area (filled)")
ax[0].scatter([], [], s=40, facecolor="none", edgecolor="0.4", label="M3 canopy area (open)")
ax[0].axhline(3.7, color="0.3", lw=1)
ax[1].axhline(1.4, color="0.3", lw=1)
ax[0].set(
    xlabel="Ur", ylabel=r"$\beta_{rc} = \beta_c / \beta_s$", title="canopy / stem (line: 3.7)"
)
ax[1].set(xlabel="Ur", ylabel=r"$\beta_{rr} = \beta_r / \beta_s$", title="root / stem (line: 1.4)")
ax[0].legend(fontsize=7)
for h_, col in DEPTH_COLOR.items():
    ax[1].scatter([], [], s=40, color=col, label=f"h = {h_} m")
ax[1].legend(fontsize=7)
fig.tight_layout()
plt.show()

- **Canopy/stem.** With the single-triangle canopy (M2 geometry, the caption formula),
  $\beta_{rc}$ runs from about 5.9 at low Ur to 3.6–3.7 at Ur ≈ 21 for h = 0.4 and
  0.5 m, and from 1.0 to 1.5 at h = 0.3 m. These are the ranges plotted in Fig. 8 for
  all three depths, **including h = 0.3 m, where only M3 was tested**. Fig. 8 therefore
  appears to use the caption's canopy area without the M3 doubling that Fig. 3a
  shows. The paper is internally inconsistent here (E7).
- **Root/stem.** $\beta_{rr}$ has the right trend (rising with Ur, falling with kh), but
  it is about **twice** the values in Fig. 8 and tends to about 2.8 rather than 1.4. With
  28 roots of 0.005 m per tree (1008 /m²) the root frontal area is fixed by Table 1.
  The factor is not resolved here (see the open questions for the author; no
  coefficient was changed).

**E4.** With the paper's limits 3.7 and 1.4 the shares are canopy 3.7/6.1 = 61 %,
root 1.4/6.1 = 23 % and stem 1/6.1 = 16 % (theory §5.6). The paper's text assigns
23 % to the stem and 16 % to the roots.

## 7. How much the geometry assumption matters

The same constant $C_D$ applied with the resolved layer profile (theory Eq. 3.4) and
with Zhang's depth-averaged area (theory Eq. 5.5) gives different $\beta$. The ratio
below is the size of the geometric assumption built into every $C_D$ in Table 3.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for c in ("M1", "M2", "M3"):
    pts = []
    for t in TEST_MATRIX:
        if c not in t.configs:
            continue
        w = sd.RegularWave(t.H, t.T, t.h)
        resolved = sd.propagate(w, VEG[c], B, drag=1.0).beta
        averaged = sd.propagate(w, VEG[c].depth_averaged(t.h), B, drag=1.0).beta
        pts.append((w.kh, resolved / averaged, t.h))
    kh_, ratio, _ = zip(*pts, strict=True)
    ax.scatter(
        kh_,
        ratio,
        s=50,
        color=COLOR[c],
        marker=MARKER[c],
        edgecolor="white",
        linewidth=1,
        label=c,
        zorder=3,
    )
    print(f"{c}: resolved / depth-averaged beta = {min(ratio):.2f} – {max(ratio):.2f}")
ax.axhline(1, color="0.5", lw=1)
ax.set(xlabel="kh", ylabel=r"$\beta_\text{resolved} / \beta_\text{depth-averaged}$")
ax.legend()
plt.show()

## 8. Summary

| Target | Result | Notes |
|---|---|---|
| Fig. 3: $A_p$, area fractions, SV, SVF | **Reproduced** | Needs the M3 canopy area doubled (E7). SVF range 0.0062–0.0168 as in Fig. 10. |
| kh, H/L, H/h, Ur ranges | **Reproduced** | |
| KC_d, Re_d ranges (E5) | **Reproduced, explained** | Table 3 uses $\bar u$; the Conclusions use $u_m$ at the still-water level. |
| Re_rv range | **Reproduced** | Within about 0.5 %; the water temperature is not stated. |
| β(M3)/β(M2) ≈ 2 | **Partial** | Only the $Re_{rv}$ closure gives about 2; the $KC_d$ and Ur closures give about 1.4. |
| β(M3)/β(M1) ≈ 7.8, 6.2, 10.7 | **Not reproduced** | Pooled closures give M1 and M3 the same $C_D$ (§4). |
| Largest β at h = 0.4 m (canopy models) | **Reproduced** | |
| β_c : β_s : β_r → 3.7 : 1 : 1.4 | **Partial** | Canopy ratio reproduced with the caption canopy area; root ratio about 2× too large. |

None of these results involved changing a coefficient to improve agreement.